<a href="https://colab.research.google.com/github/delubio/APP/blob/main/MACHINE_LEARNING_CLASSIFICACAO_DAS_ESPECIES_DE_IRES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# UNIDADE 4 - AULA 4: MACHINE LEARNING COM PYTHON
# CLASSIFICAÇÃO DAS ESPÉCIES DE IRIS COM TENSORFLOW E SCIKIT-LEARN
# ==============================================================================

# PASSO 1: IMPORTAR BIBLIOTECAS E CARREGAR DADOS

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# Fixar sementes para garantir reprodutibilidade nos resultados
np.random.seed(42)
tf.random.set_seed(42)

# Carregar o conjunto de dados Iris nativo do Scikit-Learn
iris = load_iris()
X = iris.data  # Atributos (comprimento/largura de sépalas e pétalas)
y = iris.target  # Rótulos (0: Setosa, 1: Versicolor, 2: Virginica)

# Converter em DataFrame Pandas para facilitar a exploração inicial
df_iris = pd.DataFrame(X, columns=iris.feature_names)
df_iris['especie'] = y
print("--- PRIMEIRAS 5 LINHAS DO CONJUNTO DE DADOS ---")
print(df_iris.head())
print(f"\nEspécies mapeadas: {dict(enumerate(iris.target_names))}\n")


# PASSO 2: PRÉ-PROCESSAMENTO DOS DADOS

# 2.1 Divisão dos dados em conjunto de Treinamento (80%) e Teste (20%)
# O parâmetro stratify garante que a proporção das 3 classes seja mantida em ambas as divisões
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2.2 Normalização/Padronização das características
# Redimensiona os dados para média 0 e desvio padrão 1, otimizando a convergência do TensorFlow
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Amostras de Treino: {X_train_scaled.shape[0]} | Amostras de Teste: {X_test_scaled.shape[0]}")


# PASSO 3: CONSTRUIR O MODELO (REDE NEURAL COM TENSORFLOW/KERAS)

# Construção de um modelo Perceptron Multicamadas (MLP) Sequencial
model = tf.keras.Sequential([
    # Camada de Entrada + 1ª Camada Oculta com 16 neurônios e ativação ReLU
    tf.keras.layers.Dense(16, activation='relu', input_shape=(X_train_scaled.shape[1],)),

    # 2ª Camada Oculta com 12 neurônios e ativação ReLU
    tf.keras.layers.Dense(12, activation='relu'),

    # Camada de Saída com 3 neurônios (um para cada espécie) e ativação Softmax
    tf.keras.layers.Dense(3, activation='softmax')
])

# Compilação do modelo definindo Otimizador, Função de Perda e Métricas
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # Ideal para rótulos inteiros (0, 1, 2)
    metrics=['accuracy']
)

print("\n--- ESTRUTURA DA REDE NEURAL ---")
model.summary()


# PASSO 4: TREINAR O MODELO

print("\n--- INICIANDO TREINAMENTO ---")
historico = model.fit(
    X_train_scaled,
    y_train,
    epochs=60,           # Número de iterações sobre o conjunto completo
    batch_size=8,        # Quantidade de amostras por atualização de peso
    validation_split=0.1, # 10% dos dados de treino reservados para validação rápida
    verbose=1
)


# PASSO 5: AVALIAR O MODELO

print("\n--- AVALIAÇÃO DO MODELO COM DADOS DE TESTE ---")
perda_teste, acuracia_teste = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Perda no Teste (Loss)    : {perda_teste:.4f}")
print(f"Acurácia no Teste        : {acuracia_teste * 100:.2f}%")

# Métricas detalhadas de classificação
y_pred_probs = model.predict(X_test_scaled, verbose=0)
y_pred_classes = np.argmax(y_pred_probs, axis=1)

print("\n--- RELATÓRIO DETALHADO DE CLASSIFICAÇÃO ---")
print(classification_report(y_test, y_pred_classes, target_names=iris.target_names))


# PASSO 6: FAZER PREVISÕES COM NOVAS ENTRADAS

print("--- FAZENDO PREVISÕES COM DADOS INÉDITOS ---")

# Criando 3 amostras fictícias de flores com características diferentes
amostras_novas = np.array([
    [5.1, 3.5, 1.4, 0.2],  # Esperado: setosa
    [6.2, 2.9, 4.3, 1.3],  # Esperado: versicolor
    [7.3, 2.9, 6.3, 1.8]   # Esperado: virginica
])

# Aplica a mesma escala (StandardScaler) usada no treinamento
amostras_novas_scaled = scaler.transform(amostras_novas)

# Executa a predição da rede neural
previsoes = model.predict(amostras_novas_scaled, verbose=0)
classes_previstas = np.argmax(previsoes, axis=1)

for idx, (amostra, classe_idx) in enumerate(zip(amostras_novas, classes_previstas), start=1):
    nome_especie = iris.target_names[classe_idx]
    probabilidade = previsoes[idx - 1][classe_idx] * 100
    print(f"Amostra {idx} {amostra} -> Classificada como: {nome_especie.upper()} ({probabilidade:.2f}% de confiança)")

--- PRIMEIRAS 5 LINHAS DO CONJUNTO DE DADOS ---
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   especie  
0        0  
1        0  
2        0  
3        0  
4        0  

Espécies mapeadas: {0: np.str_('setosa'), 1: np.str_('versicolor'), 2: np.str_('virginica')}

Amostras de Treino: 120 | Amostras de Teste: 30

--- ESTRUTURA DA REDE NEURAL ---


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 12)             │           204 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            39 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 323 (1.26 KB)

 Trainable params: 323 (1.26 KB)

 Non-trainable params: 0 (0.00 B)


--- INICIANDO TREINAMENTO ---
Epoch 1/60
14/14 ━━━━━━━━━━━━━━━━━━━━ 6s 56ms/step - accuracy: 0.2500 - loss: 1.2239 - val_accuracy: 0.4167 - val_loss: 1.0797
Epoch 2/60
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.5278 - loss: 1.0864 - val_accuracy: 0.8333 - val_loss: 0.9934
Epoch 3/60
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6389 - loss: 0.9939 - val_accuracy: 0.8333 - val_loss: 0.9286
Epoch 4/60
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7778 - loss: 0.9214 - val_accuracy: 0.5833 - val_loss: 0.8698
Epoch 5/60
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8241 - loss: 0.8560 - val_accuracy: 0.6667 - val_loss: 0.8130
Epoch 6/60
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8426 - loss: 0.7934 - val_accuracy: 0.6667 - val_loss: 0.7597
Epoch 7/60
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8333 - loss: 0.7327 - val_accuracy: 0.6667 - val_loss: 0.7116
Epoch 8/60
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.7870 - loss: 0.